# Churn EDA and Business Framing

This notebook translates churn data into business signals for retention strategy.

**Business question:** Which customer segments show elevated churn risk, and where should interventions be prioritized?

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

import sys

ROOT = Path.cwd().resolve().parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from churn_pipeline.pipeline import create_eda_figures, load_data

DATA_PATH = ROOT / "data" / "raw" / "telco_churn.csv"
FIG_DIR = ROOT / "reports" / "figures"

In [ ]:
df = load_data(DATA_PATH)
print("Rows, Columns:", df.shape)
df.head()

In [ ]:
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "nunique": df.nunique()
}).sort_values("missing_pct", ascending=False)
summary.head(15)

In [ ]:
churn_rate = df["Churn"].mean()
class_balance = df["Churn"].value_counts().rename({0: "Stayed", 1: "Churned"})
print(f"Overall churn rate: {churn_rate:.2%}")
class_balance

In [ ]:
contract_view = (
    df.groupby("Contract", observed=False)["Churn"]
      .agg(customers="size", churn_rate="mean")
      .assign(churn_rate=lambda x: (x["churn_rate"] * 100).round(2))
      .sort_values("churn_rate", ascending=False)
)
contract_view

In [ ]:
create_eda_figures(df, FIG_DIR)
for name in [
    "churn_rate_by_contract.png",
    "tenure_distribution_by_churn.png",
    "california_churn_hotspots.png",
]:
    print(name)
    display(Image(filename=str(FIG_DIR / name)))

## EDA Takeaways

- Contract type shows clear risk segmentation. Month-to-month plans carry notably higher churn rates.
- Lower tenure customers are concentrated in the churn class, suggesting onboarding/activation friction.
- Churn risk is not geographically uniform; city-level differences can guide targeted retention campaigns.